## STAC/ZARR 2026 OLCI inputs retrieval

This notebook allows to retrieve required inputs to prepare integration of S3 OLCI processor.


## 1. Initialization

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *
from resources.dask_utils import *

init_demo()

satellite='sentinel-3a'

# Reload the global vars again
from resources.utils import *
from resources.dask_utils import *

## 2. Find latest three valid S3A sessions from CADIP as STAC items

In [ ]:
sessions = cadip_client.search(method="GET", limit=3, max_items=3, collections="s3_sgs",
                                    stac_filter=f"platform={satellite} and cadip:delivery_push_ok=true",
                                    sortby=[ { "field": "published", "direction": "desc" } ])
for session in sessions:
  print(f"Session: {session.get_links("self")[0].get_href()}")
assert len(sessions) == 3

## 3. Find latest AUX files from ADGS as STAC items

In [ ]:
aux_files = {}

# AUX files common to all satellites
for product_type in [
    'AX___MA1_AX', 'AX___MA2_AX',                                                                             # S00__ADF_ECMWA
    'AX___MF1_AX', 'AX___MF2_AX', 'S1__AUX_WND', 'AUX_ECMWFD',                                                # S00__ADF_ECMWF
    'AX___DEM_AX',                                                                                            # S00__ADF_GETAS
    'AX___BB2_AX', 'AUX_UT1UTC_',                                                                             # S00__ADF_IERSB
    'AX___CLM_AX', 'AX___LWM_AX', 'AX___OOM_AX', 'AX___TRM_AX', 'SR___LSM_AX', 'SR_2_SURFAX', 'SR_2_MLM_AX',  # S00__ADF_WATER
]:
    aux = auxip_client.search(method="POST", limit=1, max_items=1, stac_filter={
            "op": "and",
            "args": [
                { "op": "=", "args": [ { "property": "product:type" }, product_type ] }
            ]
        }, sortby=[ { "field": "start_datetime", "direction": "desc" } ])[0]
    aux_href = aux.get_links("self")[0].get_href()
    aux_files[product_type] = aux
    print(f"{product_type}: {aux_href}")

# AUX specific per satellite
for product_type in [
    'AX___OSF_AX',  # S03X_ADF_OSFAX
    'AX___FRO_AX',  # S03X_ADF_FROAX
    'AX___FPO_AX',  # S03X_ADF_FPOAX
    'OL_2_ACP_AX',  # S03X_ADF_OLACP
    'OL_1_CAL_AX',  # S03X_ADF_OLCAL
    'OL_2_CLP_AX',  # S03X_ADF_OLCLP
    'OL_1_EO__AX',  # S03X_ADF_OLEOP
    'OL_1_INS_AX',  # S03X_ADF_OLINS
    'OL_1_CLUTAX',  # S03X_ADF_OLLUT
    'OL_2_OCP_AX',  # S03X_ADF_OLOCP
    'OL_2_PCP_AX',  # S03X_ADF_OLPCP
    'OL_2_PPP_AX',  # S03X_ADF_OLPPP
    'OL_1_PRG_AX',  # S03X_ADF_OLPRG
    'OL_1_RAC_AX',  # S03X_ADF_OLRAC
    'OL_1_SPC_AX',  # S03X_ADF_OLSPC
    'OL_2_VGP_AX',  # S03X_ADF_OLVGP
    'OL_2_WVP_AX',  # S03X_ADF_OLWVP
]:
    aux = auxip_client.search(method="POST", limit=1, max_items=1, stac_filter={
            "op": "and",
            "args": [
                { "op": "=", "args": [ { "property": "product:type" }, product_type ] },
                { "op": "=", "args": [ { "property": "platform" }, satellite ] }
            ]
        }, sortby=[ { "field": "start_datetime", "direction": "desc" } ])[0]
    aux_href = aux.get_links("self")[0].get_href()
    aux_files[product_type] = aux
    print(f"{product_type}: {aux_href}")

## 4. Create Catalog collections

In [ ]:
temporal = TemporalExtent([datetime(2026, 1, 1), datetime.now()])

sessions_coll = get_or_create_test_collection(
    collection_id="s3_sessions", description="S3 staged sessions", title="S3 sessions", temporal=temporal)

aux_coll = get_or_create_test_collection(
    collection_id="s3_aux", description="S3 staged auxiliary files", title="S3 auxiliary files", temporal=temporal)

print(f"S3 sessions collection: {sessions_coll.get_links('self')[0].get_href()}")
print(f"S3 aux collection: {aux_coll.get_links('self')[0].get_href()}")
print("STAC Browser: https://stac-browser-catalog.ops.rs-python.eu")

## 5. Stage sessions in the STAC Catalog

In [ ]:
for session in sessions:
  stage_single_item(session, sessions_coll)

## 6. Stage aux files in the STAC Catalog

In [ ]:
for product_type, aux in aux_files.items():
  stage_single_item(aux, aux_coll)